# 08 - Ingestão Silver Unificada (Missões 02, 03, 04, 05, 07)

Notebook consolidado com as 5 regras de qualidade Silver do estágio:

| Missão | Tabela | Regra | Justificativa |
| --- | --- | --- | --- |
| 02 | ecommerce_itens_pedido | FK com Silver de pedidos + tabela de espera | Item sem pedido pai = ingestão fora de ordem |
| 03 | ecommerce_produtos | preco_lista > 0 e < 5000 | Limites realistas para produtos |
| 04 | ecommerce_categorias | Alertar se nº de categorias raiz mudar entre lotes | Mudança impacta navegação do e-commerce |
| 05 | ecommerce_rastreamento_entregas | status_entrega na lista de valores do fluxo logístico | Valores fora do flow vão para quarentena |
| 07 | ecommerce_enderecos | estado deve ser UF válida (2 letras) | UF inválida bloqueia roteamento logístico |

## KPIs e Tabelas Geradas

| Missão | Tabela Silver (válidos) | Tabela Quarentena/Espera (inválidos) | KPI de Controle |
| --- | --- | --- | --- |
| 02 | silver_ecommerce_itens_pedido | itens_espera | Itens órfãos em espera |
| 03 | silver_ecommerce_produtos | quarentena_ecommerce_produtos | Taxa de rejeição por preço |
| 04 | silver_ecommerce_categorias | — | Alerta de mudança de categorias raiz |
| 05 | silver_ecommerce_rastreamento | quarentena_ecommerce_rastreamento | Taxa de rejeição por status |
| 07 | silver_ecommerce_enderecos | quarentena_ecommerce_enderecos | Taxa de rejeição por UF |

## Fluxo de Execução

1. **Setup** — Importa bibliotecas, testa ADLS (fallback UC se bloqueado), cria função `tabela_existe()` e tabela de controle de categorias raiz
2. **Missão 02** — Garante Silver de pedidos, retenta itens em espera, filtra itens novos, valida FK, grava válidos em Silver e órfãos em espera
3. **Missão 03** — Filtro incremental por sku, valida preco_lista (0 < preco < 5000), rejeitados → quarentena com motivo
4. **Missão 04** — Filtro incremental por id_categoria, conta categorias raiz, compara com lote anterior, alerta se mudou
5. **Missão 05** — Filtro incremental por id_rastreamento, valida status_entrega contra lista do fluxo logístico, rejeitados → quarentena
6. **Missão 07** — Filtro incremental por id_endereco, valida estado (UF de 2 letras), rejeitados → quarentena com motivo
7. **Resumo** — Tabela consolidada com totais de lote, Silver, quarentena e alertas de todas as missões

**Arquitetura:** ADLS (primário) + Unity Catalog/Lakehouse Federation (fallback serverless) | Persistência via MERGE INTO (upsert idempotente) + `optimizeWrite`/`autoCompact`

In [0]:
%run ./00_setup_config

In [0]:
# Importação das bibliotecas e configuração de camadas

from pyspark.sql import functions as F
from delta.tables import DeltaTable
import os

# --- Otimização Delta (alinhado com padrão Tech Lead) ---
spark.conf.set("spark.databricks.delta.properties.autoOptimize.optimizeWrite", "true")
spark.conf.set("spark.databricks.delta.properties.autoOptimize.autoCompact", "true")

# --- ADLS (primário) ---
storage_account_name = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

# --- Teste de conectividade ---
try:
    spark.read.format("parquet").options(**adls_options).load(
        f"abfss://raw@{storage_account_name}.dfs.core.windows.net/"
    ).limit(1).count()
    usar_adls = True
    print("✅ ADLS acessível")
except Exception:
    usar_adls = False
    print("⚠️ ADLS bloqueado (serverless) — usando UC + Lakehouse Federation")

# --- Função auxiliar: verifica se uma tabela existe no catálogo ---
def tabela_existe(nome_tabela):
    try:
        spark.table(nome_tabela).limit(1).count()
        return True
    except Exception:
        return False

# --- Função MERGE INTO (upsert idempotente por chave de negócio) ---
def merge_into(df, tabela_uc, chave):
    """Faz MERGE INTO (upsert) na tabela Delta pela chave de negócio.
    Garante idempotência: se o registro já existe, atualiza; se não, insere."""
    total = df.count()
    if total == 0:
        print(f"  ⚠️ {tabela_uc}: DataFrame vazio — pulando MERGE")
        return
    if not tabela_existe(tabela_uc):
        # Primeira execução: cria a tabela com append
        (df.write.format("delta")
            .option("optimizeWrite", "true")
            .option("autoCompact", "true")
            .mode("append")
            .saveAsTable(tabela_uc))
        print(f"  ✅ Delta (criação): {tabela_uc} — {total} registros inseridos")
    else:
        # Execuções subsequentes: MERGE INTO por chave
        delta_table = DeltaTable.forName(spark, tabela_uc)
        (delta_table.alias("target")
            .merge(df.alias("source"), f"target.{chave} = source.{chave}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"  ✅ Delta (MERGE): {tabela_uc} — {total} registros upserted por '{chave}'")

# --- Função append em quarentena (mantém log de rejeições) ---
def append_quarentena(df, tabela_quar):
    """Append em tabela de quarentena (mantém histórico de rejeições)."""
    total = df.count()
    if total == 0:
        print(f"  ⚠️ {tabela_quar}: nenhum registro rejeitado")
        return
    (df.write.format("delta")
        .option("optimizeWrite", "true")
        .option("autoCompact", "true")
        .mode("append")
        .saveAsTable(tabela_quar))
    print(f"  ❌ Quarentena: {tabela_quar} — {total} registros rejeitados")

# --- Tabela de controle de categorias raiz (missão 04) ---
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.controle_categorias_raiz (
        total_categorias_raiz  BIGINT,
        nomes_categorias_raiz STRING,
        dt_processamento       TIMESTAMP
    ) USING DELTA
    COMMENT 'Controle: total de categorias raiz por lote (missão 04).'
""")

print("Configuração concluída.")

---
# 🟦 Missão 02 — ecommerce_itens_pedido

**Regra:** `id_pedido` deve existir na Silver de pedidos (FK em tempo real). Itens sem pedido pai → tabela de espera.

**Justificativa:** Item sem pedido pai = ingestão fora de ordem. Guardar em tabela de espera e retentar.

| Item | Valor |
| --- | --- |
| **Origem** | `sqlserver_catalog.squad1.ecommerce_itens_pedido` |
| **Silver** | `workspace.default.silver_ecommerce_itens_pedido` |
| **Espera** | `workspace.default.itens_espera` |
| **Pré-requisito** | Silver de pedidos (`silver_ecommerce_pedidos`) deve existir |

**Fluxo:**
1. Garante que a Silver de pedidos existe (cria carga inicial se necessário)
2. Retenta itens em espera (recupera se o pedido pai chegou na Silver)
3. Filtra apenas itens novos (anti-join por `id_item_pedido`)
4. Valida FK: inner join com Silver de pedidos (válidos) × left anti (órfãos)
5. Válidos → Silver (MERGE INTO por `id_item_pedido`) | Órfãos → tabela de espera (MERGE por `id_item_pedido`)

In [0]:
# ============================================================================
# Missão 02: ecommerce_itens_pedido — FK com Silver de pedidos + tabela de espera
# ============================================================================

# ⚠️ Confirmar com o Kauan: itens_pedido está no squad1, não no squad2
tabela_origem_itens = "sqlserver_catalog.squad1.ecommerce_itens_pedido"
tabela_origem_pedidos = "sqlserver_catalog.squad2.ecommerce_pedidos"
tabela_silver_pedidos = "workspace.default.silver_ecommerce_pedidos"
tabela_silver_itens = "workspace.default.silver_ecommerce_itens_pedido"
tabela_espera = "workspace.default.itens_espera"

# --- Pré-requisito: garantir que a Silver de pedidos existe ---
if not tabela_existe(tabela_silver_pedidos):
    print("⚠️ Silver de pedidos não existe — criando carga inicial...")
    df_pedidos_origem = spark.table(tabela_origem_pedidos)
    merge_into(df_pedidos_origem.withColumn("silver_processed_at", F.current_timestamp()),
               tabela_silver_pedidos, "id_pedido")
    total_pedidos_silver = df_pedidos_origem.count()
    print(f"✅ Silver de pedidos criada: {total_pedidos_silver} registros")
else:
    total_pedidos_silver = spark.table(tabela_silver_pedidos).count()
    print(f"✅ Silver de pedidos já existe: {total_pedidos_silver} registros")

# --- Passo 1: Retentativa de itens em espera ---
if tabela_existe(tabela_espera):
    df_espera = spark.table(tabela_espera)
    total_espera = df_espera.count()
    if total_espera > 0:
        print(f"📋 Retentando {total_espera} item(ns) em espera...")
        df_silver_pedidos_ids = spark.table(tabela_silver_pedidos).select("id_pedido").distinct()
        df_espera_recuperados = df_espera.join(df_silver_pedidos_ids, "id_pedido", "inner")
        n_recuperados = df_espera_recuperados.count()
        if n_recuperados > 0:
            merge_into(df_espera_recuperados.drop("dt_espera").withColumn("silver_processed_at", F.current_timestamp()),
                       tabela_silver_itens, "id_item_pedido")
            df_espera_restante = df_espera.join(df_espera_recuperados.select("id_item_pedido"), "id_item_pedido", "left_anti")
            # Remove os recuperados da tabela de espera: coleta IDs e executa DELETE via SQL
            ids_recuperados_str = ",".join([str(r["id_item_pedido"]) for r in df_espera_recuperados.select("id_item_pedido").collect()])
            spark.sql(f"DELETE FROM {tabela_espera} WHERE id_item_pedido IN ({ids_recuperados_str})")
            print(f"  ✅ {n_recuperados} item(ns) recuperados da espera → Silver")
            print(f"  ⏳ {total_espera - n_recuperados} item(ns) continuam em espera")
        else:
            print(f"  ⏳ Nenhum item recuperado — {total_espera} item(ns) continuam em espera")
    else:
        print("📋 Tabela de espera está vazia.")
else:
    print("📋 Tabela de espera ainda não existe (primeira execução).")

# --- Passo 2: Ler itens da origem e filtrar apenas os novos ---
df_itens_origem = spark.table(tabela_origem_itens)
print(f"📊 Total de itens na origem: {df_itens_origem.count()}")

# Filtra apenas itens que ainda não estão na Silver de itens nem na espera
ids_ja_processados = None
if tabela_existe(tabela_silver_itens):
    ids_ja_processados = spark.table(tabela_silver_itens).select("id_item_pedido").distinct()
if tabela_existe(tabela_espera):
    ids_espera = spark.table(tabela_espera).select("id_item_pedido").distinct()
    if ids_ja_processados is not None:
        ids_ja_processados = ids_ja_processados.union(ids_espera)
    else:
        ids_ja_processados = ids_espera

if ids_ja_processados is not None:
    df_itens_novos = df_itens_origem.join(ids_ja_processados, "id_item_pedido", "left_anti")
else:
    df_itens_novos = df_itens_origem

total_itens_lote = df_itens_novos.count()
print(f"📊 Itens novos (não processados): {total_itens_lote}")

# --- Passo 3: Validar FK ---
if total_itens_lote > 0:
    df_silver_pedidos_ids = spark.table(tabela_silver_pedidos).select("id_pedido").distinct()

    # Inner join: itens com pedido pai na Silver (válidos)
    df_itens_validos = df_itens_novos.join(df_silver_pedidos_ids, "id_pedido", "inner")

    # Left anti join: itens sem pedido pai (órfãos)
    df_itens_orfaos = df_itens_novos.join(df_silver_pedidos_ids, "id_pedido", "left_anti")

    n_itens_validos = df_itens_validos.count()
    n_itens_orfaos = df_itens_orfaos.count()

    print(f"✅ Itens com FK válida: {n_itens_validos}")
    print(f"⏳ Itens órfãos (sem pedido pai): {n_itens_orfaos}")

    # --- Passo 4: Gravar Silver de itens (válidos) e tabela de espera (órfãos) ---
    if n_itens_validos > 0:
        merge_into(df_itens_validos.withColumn("silver_processed_at", F.current_timestamp()),
                   tabela_silver_itens, "id_item_pedido")
    else:
        print("Nenhum item válido para gravar na Silver.")

    if n_itens_orfaos > 0:
        merge_into(df_itens_orfaos.withColumn("dt_espera", F.current_timestamp()),
                   tabela_espera, "id_item_pedido")
    else:
        print("Nenhum item órfão para colocar em espera.")
else:
    n_itens_validos = 0
    n_itens_orfaos = 0
    print("Nenhum item novo para processar.")

---
# 🟦 Missão 03 — ecommerce_produtos

**Regra:** `preco_lista` deve ser > 0 e < 5000

**Justificativa:** Limites realistas para produtos.

| Item | Valor |
| --- | --- |
| **Origem** | `sqlserver_catalog.squad2.ecommerce_produtos` |
| **Silver** | `workspace.default.silver_ecommerce_produtos` |
| **Quarentena** | `workspace.default.quarentena_ecommerce_produtos` |
| **Chave incremental** | `sku` |

**Fluxo:**
1. Filtro incremental: anti-join por `sku` (só processa produtos novos)
2. Validações técnicas + regra de negócio:
   - `SKU_NULO` — SKU nulo
   - `SKU_TAMANHO_INVALIDO` — comprimento fora de 5-60 caracteres
   - `IS_ATIVO_NULO` — is_ativo nulo
   - `PRECO_NULO` — preço nulo
   - `PRECO_MENOR_OU_IGUAL_ZERO` — preço ≤ 0
   - `PRECO_MAIOR_OU_IGUAL_5000` — preço ≥ 5000
3. Válidos → Silver (MERGE INTO por `sku`) | Rejeitados → Quarentena (append com `motivo_rejeicao`)

In [0]:
# ============================================================================
# Missão 03: ecommerce_produtos — preco_lista > 0 e < 5000
# ============================================================================

PRECO_MIN = 0.0
PRECO_MAX = 5000.0

tabela_origem_prod = "sqlserver_catalog.squad2.ecommerce_produtos"
tabela_silver_prod = "workspace.default.silver_ecommerce_produtos"
tabela_quar_prod = "workspace.default.quarentena_ecommerce_produtos"

df_prod_origem = spark.table(tabela_origem_prod)

# Filtro incremental: anti-join por sku
if tabela_existe(tabela_silver_prod):
    df_prod_ja = spark.table(tabela_silver_prod).select("sku").distinct()
    df_prod_lote = df_prod_origem.join(df_prod_ja, "sku", "left_anti")
else:
    df_prod_lote = df_prod_origem

total_prod_lote = df_prod_lote.count()
print(f"📊 Produtos no lote: {total_prod_lote}")

if total_prod_lote > 0:
    colunas_prod = df_prod_lote.columns
    df_prod_t = df_prod_lote.withColumn("preco_lista_num", F.col("preco_lista").cast("double"))
    # Classifica rejeição: validações técnicas + regra de negócio (alinhado com Dupla 01)
    df_prod_t = df_prod_t.withColumn("motivo_rejeicao",
        F.when(F.col("sku").isNull(), F.lit("SKU_NULO"))
         .when((F.length(F.col("sku")) < 5) | (F.length(F.col("sku")) > 60), F.lit("SKU_TAMANHO_INVALIDO"))
         .when(F.col("is_ativo").isNull(), F.lit("IS_ATIVO_NULO"))
         .when(F.col("preco_lista_num").isNull(), F.lit("PRECO_NULO"))
         .when(F.col("preco_lista_num") <= PRECO_MIN, F.lit("PRECO_MENOR_OU_IGUAL_ZERO"))
         .when(F.col("preco_lista_num") >= PRECO_MAX, F.lit("PRECO_MAIOR_OU_IGUAL_5000")))

    df_prod_validos = df_prod_t.filter(F.col("motivo_rejeicao").isNull())
    df_prod_quar = df_prod_t.filter(F.col("motivo_rejeicao").isNotNull())

    n_prod_validos = df_prod_validos.count()
    n_prod_quar = df_prod_quar.count()

    # Gravar (MERGE INTO para Silver, append para quarentena)
    if n_prod_quar > 0:
        append_quarentena(df_prod_quar.select(*colunas_prod, "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_quar_prod)

    if n_prod_validos > 0:
        merge_into(df_prod_validos.drop("preco_lista_num", "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_silver_prod, "sku")

    print(f"✅ Produtos válidos: {n_prod_validos} → Silver")
    print(f"❌ Produtos rejeitados: {n_prod_quar} → Quarentena")
    if n_prod_quar > 0:
        display(df_prod_quar.groupBy("motivo_rejeicao").count())
else:
    n_prod_validos = 0
    n_prod_quar = 0
    print("Nenhum produto novo.")

---
# 🟦 Missão 04 — ecommerce_categorias

**Regra:** Alertar se o número total de categorias raiz mudar entre lotes

**Justificativa:** Mudança impacta navegação do e-commerce.

| Item | Valor |
| --- | --- |
| **Origem** | `sqlserver_catalog.squad2.ecommerce_categorias` |
| **Silver** | `workspace.default.silver_ecommerce_categorias` |
| **Controle** | `workspace.default.controle_categorias_raiz` |
| **Chave incremental** | `id_categoria` |

**Fluxo:**
1. Filtro incremental: anti-join por `id_categoria`
2. Validações técnicas:
   - `ID_CATEGORIA_NULO` — id_categoria nulo
   - `NOME_CATEGORIA_NULO` — nome_categoria nulo ou vazio
3. Conta categorias raiz (`id_categoria_pai IS NULL`) no lote atual
4. Compara com o último registro na tabela de controle
5. Se diferente → emite alerta com categorias adicionadas/removidas
6. Válidos → Silver (MERGE INTO por `id_categoria`) | Rejeitados → Quarentena

In [0]:
# ============================================================================
# Missão 04: ecommerce_categorias — alerta de categorias raiz
# ============================================================================

tabela_origem_cat = "sqlserver_catalog.squad2.ecommerce_categorias"
tabela_silver_cat = "workspace.default.silver_ecommerce_categorias"
tabela_controle_raiz = "workspace.default.controle_categorias_raiz"

df_cat_origem = spark.table(tabela_origem_cat)

# Filtro incremental: anti-join por id_categoria
if tabela_existe(tabela_silver_cat):
    df_cat_ja = spark.table(tabela_silver_cat).select("id_categoria").distinct()
    df_cat_lote = df_cat_origem.join(df_cat_ja, "id_categoria", "left_anti")
else:
    df_cat_lote = df_cat_origem

total_cat_lote = df_cat_lote.count()

# Conta categorias raiz (id_categoria_pai IS NULL)
df_raiz = df_cat_origem.filter(F.col("id_categoria_pai").isNull())
total_raiz_atual = df_raiz.count()
nomes_raiz_atual = [r["nome_categoria"] for r in df_raiz.select("nome_categoria").orderBy("nome_categoria").collect()]

print(f"📊 Categorias no lote: {total_cat_lote}")
print(f"📊 Categorias raiz: {total_raiz_atual}")

# Compara total de categorias raiz com o lote anterior; se diferente, emite alerta
alerta_raiz = False
if tabela_existe(tabela_controle_raiz):
    ultima = spark.table(tabela_controle_raiz).orderBy(F.col("dt_processamento").desc()).limit(1).collect()
    if ultima:
        total_raiz_anterior = ultima[0]["total_categorias_raiz"]
        nomes_raiz_anterior = ultima[0]["nomes_categorias_raiz"]
        if total_raiz_atual != total_raiz_anterior:
            alerta_raiz = True
            set_ant = set(nomes_raiz_anterior.split(", ")) if nomes_raiz_anterior else set()
            set_atu = set(nomes_raiz_atual)
            print("=" * 60)
            print("🚨 [ALERTA] NÚMERO DE CATEGORIAS RAIZ MUDOU!")
            print(f"   Anterior: {total_raiz_anterior} | Atual: {total_raiz_atual}")
            if set_atu - set_ant:
                print(f"   ADICIONADAS: {set_atu - set_ant}")
            if set_ant - set_atu:
                print(f"   REMOVIDAS: {set_ant - set_atu}")
            print("=" * 60)
        else:
            print("✅ Número de categorias raiz inalterado.")

# Gravar Silver (MERGE INTO) + quarentena + atualizar controle
if total_cat_lote > 0:
    # Validação técnica: id_categoria e nome_categoria obrigatórios (alinhado com Dupla 01)
    df_cat_t = df_cat_lote.withColumn("motivo_rejeicao",
        F.when(F.col("id_categoria").isNull(), F.lit("ID_CATEGORIA_NULO"))
         .when(F.col("nome_categoria").isNull() | (F.trim(F.col("nome_categoria")) == ""), F.lit("NOME_CATEGORIA_NULO")))

    df_cat_validos = df_cat_t.filter(F.col("motivo_rejeicao").isNull())
    df_cat_quar = df_cat_t.filter(F.col("motivo_rejeicao").isNotNull())

    n_cat_validos = df_cat_validos.count()
    n_cat_quar = df_cat_quar.count()
    colunas_cat = df_cat_lote.columns

    if n_cat_quar > 0:
        append_quarentena(df_cat_quar.select(*colunas_cat, "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()),
            "workspace.default.quarentena_ecommerce_categorias")

    if n_cat_validos > 0:
        merge_into(df_cat_validos.drop("motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_silver_cat, "id_categoria")
else:
    n_cat_validos = 0
    n_cat_quar = 0

nomes_str = ", ".join(nomes_raiz_atual)
spark.sql(f"""
    INSERT INTO {tabela_controle_raiz} (total_categorias_raiz, nomes_categorias_raiz, dt_processamento)
    VALUES ({total_raiz_atual}, '{nomes_str}', current_timestamp())
""")
print(f"✅ Controle atualizado: {total_raiz_atual} categorias raiz")

---
# 🟦 Missão 05 — ecommerce_rastreamento_entregas

**Regra:** `status_entrega` deve estar na lista de valores do fluxo logístico

**Justificativa:** Valores fora do flow vão para quarentena.

| Item | Valor |
| --- | --- |
| **Origem** | `sqlserver_catalog.squad2.ecommerce_rastreamento_entregas` |
| **Silver** | `workspace.default.silver_ecommerce_rastreamento` |
| **Quarentena** | `workspace.default.quarentena_ecommerce_rastreamento` |
| **Chave incremental** | `id_rastreamento` |

**Fluxo:**
1. Filtro incremental: anti-join por `id_rastreamento`
2. Normaliza `status_entrega` (lower + trim) e valida contra lista do fluxo logístico
3. Rejeitados recebem motivo: `STATUS_NULO` ou `STATUS_FORA_DO_FLOW`
4. Válidos → Silver (MERGE INTO por `id_rastreamento`) | Rejeitados → Quarentena (append com `motivo_rejeicao`)

In [0]:
# ============================================================================
# Missão 05: ecommerce_rastreamento_entregas — status_entrega válido
# ============================================================================

# ⚠️ Confirmar lista com o professor
STATUS_VALIDOS = [
    "aguardando separacao", "em separacao", "separado", "em transito",
    "entregue", "falha", "devolvido",
    "aguardando_separacao", "em_separacao", "em_transito"
]

tabela_origem_rast = "sqlserver_catalog.squad2.ecommerce_rastreamento_entregas"
tabela_silver_rast = "workspace.default.silver_ecommerce_rastreamento"
tabela_quar_rast = "workspace.default.quarentena_ecommerce_rastreamento"

df_rast_origem = spark.table(tabela_origem_rast)

# Filtro incremental: anti-join por id_rastreamento
if tabela_existe(tabela_silver_rast):
    df_rast_ja = spark.table(tabela_silver_rast).select("id_rastreamento").distinct()
    df_rast_lote = df_rast_origem.join(df_rast_ja, "id_rastreamento", "left_anti")
else:
    df_rast_lote = df_rast_origem

total_rast_lote = df_rast_lote.count()
print(f"📊 Rastreamentos no lote: {total_rast_lote}")

if total_rast_lote > 0:
    colunas_rast = df_rast_lote.columns
    # Normaliza status (lower + trim) e classifica rejeição: nulo ou fora do fluxo logístico
    df_rast_t = df_rast_lote.withColumn("status_normalizado", F.lower(F.trim(F.col("status_entrega"))))
    df_rast_t = df_rast_t.withColumn("motivo_rejeicao",
        F.when(F.col("status_entrega").isNull(), F.lit("STATUS_NULO"))
         .when(~F.col("status_normalizado").isin(STATUS_VALIDOS), F.lit("STATUS_FORA_DO_FLOW")))

    df_rast_validos = df_rast_t.filter(F.col("motivo_rejeicao").isNull())
    df_rast_quar = df_rast_t.filter(F.col("motivo_rejeicao").isNotNull())

    n_rast_validos = df_rast_validos.count()
    n_rast_quar = df_rast_quar.count()

    if n_rast_quar > 0:
        append_quarentena(df_rast_quar.select(*colunas_rast, "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_quar_rast)

    if n_rast_validos > 0:
        merge_into(df_rast_validos.drop("status_normalizado", "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_silver_rast, "id_rastreamento")

    print(f"✅ Rastreamentos válidos: {n_rast_validos} → Silver")
    print(f"❌ Rastreamentos rejeitados: {n_rast_quar} → Quarentena")
else:
    n_rast_validos = 0
    n_rast_quar = 0
    print("Nenhum rastreamento novo.")

---
# 🟦 Missão 07 — ecommerce_enderecos

**Regra:** `estado` deve ser UF válida (2 letras)

**Justificativa:** UF inválida bloqueia roteamento logístico.

| Item | Valor |
| --- | --- |
| **Origem** | `sqlserver_catalog.squad2.ecommerce_enderecos` |
| **Silver** | `workspace.default.silver_ecommerce_enderecos` |
| **Quarentena** | `workspace.default.quarentena_ecommerce_enderecos` |
| **Chave incremental** | `id_endereco` |

**Fluxo:**
1. Filtro incremental: anti-join por `id_endereco`
2. Normaliza `estado` (upper + trim) e valida:
   - `ESTADO_NULO` — estado nulo
   - `UF_FORMATO_INVALIDO` — não tem 2 letras
   - `UF_NAO_RECONHECIDA` — não está na lista de 27 UFs válidas
3. Válidos → Silver (MERGE INTO por `id_endereco`) | Rejeitados → Quarentena (append com `motivo_rejeicao`)

In [0]:
# ============================================================================
# Missão 07: ecommerce_enderecos — estado deve ser UF válida
# ============================================================================

UFS_VALIDAS = [
    "AC", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE",
    "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP",
    "SE", "TO"
]

tabela_origem_end = "sqlserver_catalog.squad2.ecommerce_enderecos"
tabela_silver_end = "workspace.default.silver_ecommerce_enderecos"
tabela_quar_end = "workspace.default.quarentena_ecommerce_enderecos"

df_end_origem = spark.table(tabela_origem_end)

# Filtro incremental: anti-join por id_endereco
if tabela_existe(tabela_silver_end):
    df_end_ja = spark.table(tabela_silver_end).select("id_endereco").distinct()
    df_end_lote = df_end_origem.join(df_end_ja, "id_endereco", "left_anti")
else:
    df_end_lote = df_end_origem

total_end_lote = df_end_lote.count()
print(f"📊 Enderecos no lote: {total_end_lote}")

if total_end_lote > 0:
    colunas_end = df_end_lote.columns
    # Normaliza estado (upper + trim) e valida formato (2 letras) e UF válida
    df_end_t = df_end_lote.withColumn("estado_normalizado", F.upper(F.trim(F.col("estado"))))
    df_end_t = df_end_t.withColumn("motivo_rejeicao",
        F.when(F.col("estado").isNull(), F.lit("ESTADO_NULO"))
         .when(F.length(F.col("estado_normalizado")) != 2, F.lit("UF_FORMATO_INVALIDO"))
         .when(~F.col("estado_normalizado").isin(UFS_VALIDAS), F.lit("UF_NAO_RECONHECIDA")))

    df_end_validos = df_end_t.filter(F.col("motivo_rejeicao").isNull())
    df_end_quar = df_end_t.filter(F.col("motivo_rejeicao").isNotNull())

    n_end_validos = df_end_validos.count()
    n_end_quar = df_end_quar.count()

    if n_end_quar > 0:
        append_quarentena(df_end_quar.select(*colunas_end, "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_quar_end)

    if n_end_validos > 0:
        merge_into(df_end_validos.drop("estado_normalizado", "motivo_rejeicao")
            .withColumn("silver_processed_at", F.current_timestamp()), tabela_silver_end, "id_endereco")

    print(f"✅ Enderecos válidos: {n_end_validos} → Silver")
    print(f"❌ Enderecos rejeitados: {n_end_quar} → Quarentena")
else:
    n_end_validos = 0
    n_end_quar = 0
    print("Nenhum endereço novo.")

---
# 📊 Resumo Geral

Consolida os resultados de todas as missões em uma única tabela:

* **Lote** — registros novos processados nesta execução
* **Silver** — registros válidos gravados na camada Silver
* **Quarentena** — registros rejeitados (com `motivo_rejeicao`)
* **Alerta** — indica se houve alerta de categorias raiz (missão 04)

Também lista o total de registros em cada tabela Silver e de quarentena criada.

In [0]:
# ============================================================================
# Resumo geral — todas as missões
# ============================================================================

print("=" * 60)
print("RESUMO GERAL — INGESTÃO SILVER UNIFICADA")
print("=" * 60)

resumo = [
    ("02", "itens_pedido", total_itens_lote, n_itens_validos, n_itens_orfaos),
    ("03", "produtos", total_prod_lote, n_prod_validos, n_prod_quar),
    ("04", "categorias", total_cat_lote, n_cat_validos, n_cat_quar),
    ("05", "rastreamento", total_rast_lote, n_rast_validos, n_rast_quar),
    ("07", "enderecos", total_end_lote, n_end_validos, n_end_quar),
]

print(f"{'Missão':<8} {'Tabela':<15} {'Lote':<8} {'Silver':<8} {'Quarentena':<10} {'Alerta':<8}")
print("-" * 60)
for missao, tabela, lote, silver, quar in resumo:
    alerta = "SIM 🚨" if (missao == "04" and alerta_raiz) else "NÃO"
    print(f"{missao:<8} {tabela:<15} {lote:<8} {silver:<8} {quar:<10} {alerta:<8}")

print()
print("Tabelas Silver criadas:")
tabelas_silver = [
    "workspace.default.silver_ecommerce_pedidos",
    "workspace.default.silver_ecommerce_itens_pedido",
    "workspace.default.silver_ecommerce_produtos",
    "workspace.default.silver_ecommerce_categorias",
    "workspace.default.silver_ecommerce_rastreamento",
    "workspace.default.silver_ecommerce_enderecos",
]
for t in tabelas_silver:
    if tabela_existe(t):
        total = spark.table(t).count()
        print(f"  ✅ {t}: {total} registros")
    else:
        print(f"  ⚠️ {t}: não existe")

print()
print("Tabelas de quarentena:")
tabelas_quar = [
    "workspace.default.itens_espera",
    "workspace.default.quarentena_ecommerce_produtos",
    "workspace.default.quarentena_ecommerce_categorias",
    "workspace.default.quarentena_ecommerce_rastreamento",
    "workspace.default.quarentena_ecommerce_enderecos",
]
for t in tabelas_quar:
    if tabela_existe(t):
        total = spark.table(t).count()
        print(f"  ✅ {t}: {total} registros")
    else:
        print(f"  ⚠️ {t}: não existe")